# Pipeline complet — exécution manuelle

Ce notebook déroule tout le pipeline du projet étape par étape, pour le tester/déboguer manuellement sans passer par le cron GitHub Actions ni HuggingFace Jobs :

1. Génération + nettoyage des données (`make_dataset.py`)
2. Données météo externes (avec repli hors ligne si pas d'accès réseau)
3. Feature engineering (démo isolée du `FeatureEngineer`)
4. Entraînement (`train.py`)
5. Inférence batch (`predict_model.py`)
6. Test de l'API FastAPI (in-process, sans lancer `uvicorn`)

À lancer avec le kernel Python de l'environnement du projet (`pip install -r requirements.txt` au préalable).

In [ ]:
import os
from pathlib import Path

# Le notebook vit dans notebooks/ — on se replace à la racine du projet
# pour que tous les chemins relatifs de config.yaml (data/raw, models/...) soient corrects.
if Path.cwd().name == "notebooks":
    os.chdir("..")

print("Répertoire de travail :", Path.cwd())

## 1. Génération + nettoyage des données

Équivalent de `python -m src.data.make_dataset`, mais étape par étape pour inspecter chaque résultat intermédiaire.

In [ ]:
from src.data.make_dataset import load_raw_data, clean_data, RAW_DATA_DIR

df_raw = load_raw_data(RAW_DATA_DIR)
print(f"{len(df_raw)} lignes brutes")
df_raw.head()

In [ ]:
df_clean = clean_data(df_raw)
print(f"{len(df_clean)} lignes après nettoyage structurel")
df_clean.describe(include="all")

## 2. Données météo externes

Appelle la vraie API Open-Meteo (nécessite un accès réseau). Si l'appel échoue (pas de réseau, API indisponible), on retombe automatiquement sur des valeurs météo factices pour pouvoir continuer le notebook hors ligne.

In [ ]:
from src.data.make_dataset import fetch_external_data, merge_external_data

try:
    external_df = fetch_external_data(df_clean)
    df_merged = merge_external_data(df_clean, external_df)
    print("Données météo récupérées via l'API Open-Meteo.")
except Exception as e:
    print(f"Appel API météo impossible ({e}) — utilisation de valeurs factices pour continuer hors ligne.")
    df_merged = df_clean.copy()
    df_merged["temperature_max"] = 20.0
    df_merged["temperature_min"] = 10.0
    df_merged["precipitation"] = 0.0

df_merged[["date", "region", "temperature_max", "temperature_min", "precipitation"]].head()

## 3. Sauvegarde du dataset traité

In [ ]:
from src.data.make_dataset import save_processed_data, PROCESSED_DATA_DIR

output_file = save_processed_data(df_merged, PROCESSED_DATA_DIR)
print(f"Sauvegardé dans {output_file}")

## 4. Feature engineering — démo isolée

Aperçu de ce que `FeatureEngineer` calcule, indépendamment du reste du pipeline sklearn.

In [ ]:
from src.models.feature_engineering import FeatureEngineer

fe = FeatureEngineer()
demo = fe.fit_transform(df_merged.drop(columns=["cible"]).head())
demo[["age", "revenu", "revenu_par_age", "revenu_par_annee_anciennete", "tranche_age", "categorie_region"]]

## 5. Entraînement

Construit le pipeline complet (`FeatureEngineer` + `ColumnTransformer` + `LogisticRegression`), l'entraîne, affiche les métriques.

In [ ]:
from src.models.train import load_data, build_model, train as train_model

X, y = load_data()
model = build_model()
trained_model, metrics = train_model(model, X, y)

metrics

## 6. Sauvegarde locale du modèle

Nécessaire pour tester l'inférence et l'API en local (source `"local"`), sans dépendre de MLflow ou du Hub HuggingFace.

In [ ]:
import joblib
from src.config import load_config

CONFIG = load_config()
model_dir = Path(CONFIG["paths"]["models"])
model_dir.mkdir(parents=True, exist_ok=True)

model_path = model_dir / CONFIG["paths"]["model_filename"]
joblib.dump(trained_model, model_path)
print(f"Modèle sauvegardé dans {model_path}")

## 7. Inférence batch (`predict_model.py`)

Équivalent de `python -m src.models.predict_model --source local`, sur quelques observations du dataset.

In [ ]:
from src.models.predict_model import load_model, predict

loaded_model = load_model(source="local")
sample = X.sample(5, random_state=1)
predictions = predict(loaded_model, sample)
predictions[["prediction", "probabilite"]]

## 8. Test de l'API FastAPI (in-process)

Teste l'API sans lancer `uvicorn` séparément, via `TestClient` — utile pour valider rapidement les endpoints pendant le développement.

Pour un vrai serveur accessible depuis l'extérieur du notebook : `make api` dans un terminal, puis `make streamlit` dans un autre.

In [ ]:
from fastapi.testclient import TestClient
from src.api import model_loader

# Force la source "local" pour ce test (le modèle vient d'être sauvegardé à l'étape 6)
model_loader.CONFIG["api"]["model_source"] = "local"

from src.api.main import app

client = TestClient(app)

print(client.get("/health").json())
print(client.get("/model/info").json())

In [ ]:
payload = {
    "age": 35,
    "revenu": 42000,
    "anciennete_mois": 18,
    "categorie": "A",
    "region": "Nord",
    # "date" omis -> l'API utilise aujourd'hui et récupère la météo elle-même
}

response = client.post("/predict", json=payload)
print(response.status_code)
response.json()

## Pour aller plus loin

- **Lancer les vrais services** : `make api` (port 8000, docs sur `/docs`) puis `make streamlit` (port 8501) dans deux terminaux séparés, ou `docker compose up api streamlit`.
- **Lancer la suite de tests complète** : `make test` (ou `pytest tests/`) — exclut par défaut les tests d'intégration réseau (`pytest tests/ -m integration` pour les inclure).
- **Déclencher l'entraînement complet en conditions réelles** (HuggingFace Jobs + MLflow) : `python trigger_job.py` (nécessite les secrets configurés, voir le README).